[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/badaouihakimou/machine-learning-notebooks/blob/main/13_numpy_broadcasting.ipynb)



# NumPy : le broadcasting

C'est la dernière notion NumPy, et la plus rentable. Le broadcasting est la règle
qui permet d'additionner ou de multiplier des tableaux de formes différentes,
sans boucle et sans les recopier.

On l'a déjà utilisé sans le nommer : `A + 2`, `X - moyennes`, `A / A.sum(axis=1,
keepdims=True)`. Ce notebook explique la règle exacte et surtout l'erreur
silencieuse qu'elle provoque quand on ne la maîtrise pas.

## Le plan

| Section | Le sujet |
|---|---|
| 1 | Le cas le plus simple : tableau et scalaire |
| 2 | La règle du broadcasting, en trois points |
| 3 | Étendre une ligne, étendre une colonne |
| 4 | Ligne × colonne : le produit extérieur |
| 5 | Le piège silencieux : `(n, 1)` contre `(n,)` |
| 6 | Applications concrètes |

## Le piège annoncé

Soustraire un tableau `(1000,)` d'un tableau `(1000, 1)` ne lève aucune
erreur. Le résultat est un tableau `(1000, 1000)` un million de valeurs là où
on en attendait mille. C'est la section 5, et c'est l'erreur la plus coûteuse de
tout le bloc NumPy.

Prérequis : les notebooks 10 à 12, en particulier le `(3,)` contre `(3, 1)`.

## 1. Le cas le plus simple

Additionner deux tableaux de même forme se fait case par case.

In [1]:
import numpy as np
np.random.seed(0)
A = np.random.randint(0, 10, (2, 3))
B = np.ones((2, 3))

print('A :'); print(A)
print()
print('A + B :'); print(A + B)

A :
[[5 0 3]
 [3 7 9]]

A + B :
[[ 6.  1.  4.]
 [ 4.  8. 10.]]


Rien de nouveau : même forme, addition terme à terme.

Mais NumPy sait aussi additionner un tableau et un simple nombre :

In [2]:
print(A + 2)

[[ 7  2  5]
 [ 5  9 11]]


Le 2 est ajouté à chaque case. C'est déjà du broadcasting : le scalaire est
traité comme s'il avait été étendu à la forme de `A`.

On l'utilise depuis le notebook 10 sans y penser. La question est : jusqu'où
cette extension automatique va-t-elle ?

## 2. La règle du broadcasting

Quand les formes diffèrent, NumPy tente d'étendre le plus petit tableau. Il suit
une règle précise, en comparant les formes de droite à gauche.

Pour chaque dimension, deux tableaux sont compatibles si :

- elles sont égales, ou
- l'une des deux vaut 1.

Si une dimension vaut 1, elle est étirée pour correspondre à l'autre. Si aucune
des deux conditions n'est remplie, NumPy lève une erreur.

### Quand ça échoue

In [3]:
A = np.random.randint(0, 10, (2, 3))
B = np.ones((2, 2))

try:
    A + B
except ValueError as e:
    print('ValueError :', str(e)[:75])

ValueError : operands could not be broadcast together with shapes (2,3) (2,2) 


`(2, 3)` et `(2, 2)` : la dernière dimension vaut 3 d'un côté, 2 de l'autre.
Différentes, et aucune ne vaut 1. Incompatibles.

Le message est explicite il donne les deux formes. C'est le premier réflexe :
lire les formes dans le message d'erreur.

### Quand ça marche

In [4]:
A = np.random.randint(0, 10, (2, 3))
B = np.ones((2, 1)) # une seule colonne

print('A :', A.shape)
print('B :', B.shape)
print('A + B :', (A + B).shape)
print()
print(A + B)

A : (2, 3)
B : (2, 1)
A + B : (2, 3)

[[9. 9. 2.]
 [7. 8. 8.]]


`(2, 3)` et `(2, 1)` : la dernière dimension vaut 3 et 1. Le 1 est étiré à 3.
Compatible.

Concrètement, la colonne unique de `B` est répétée sur les trois colonnes,
puis l'addition se fait case par case.

## 3. Étendre une ligne, étendre une colonne

La différence entre `(2, 1)` et `(1, 2)` illustre toute la règle.

In [5]:
A = np.random.randint(0, 10, (2, 3))

colonne = np.ones((2, 1))
print('A + colonne (2,1) :', (A + colonne).shape)
print(A + colonne)

A + colonne (2,1) : (2, 3)
[[ 9.  2.  6.]
 [10.  9. 10.]]


Une colonne `(2, 1)` s'étire horizontalement : elle est copiée sur toutes les
colonnes de `A`.

In [6]:
ligne = np.ones((1, 3))
print('A + ligne (1,3) :', (A + ligne).shape)
print(A + ligne)

A + ligne (1,3) : (2, 3)
[[ 9.  2.  6.]
 [10.  9. 10.]]


Une ligne `(1, 3)` s'étire verticalement : copiée sur toutes les lignes.

C'est exactement ce qu'on faisait dans la standardisation du notebook précédent.
`X - moyennes`, avec `X` de forme `(100, 4)` et `moyennes` de forme `(4,)` :
chaque ligne se voit soustraire le même vecteur de moyennes.

### Le dessin mental

```
colonne (2,1) ligne (1,3)
  [a] -> [a a a] [x y z] -> [x y z]
  [b] [b b b] [x y z]
```

Une dimension à 1 est le signal « répète-moi dans cette direction ».

## 4. Ligne × colonne : le produit extérieur

Le cas le plus spectaculaire : une colonne et une ligne combinées produisent une
matrice complète.

In [7]:
colonne = np.array([[1], [2], [3], [4]]) # (4, 1)
ligne = np.array([[10, 20, 30]]) # (1, 3)

print('colonne :', colonne.shape)
print('ligne :', ligne.shape)
print()
resultat = colonne + ligne
print('somme :', resultat.shape)
print(resultat)

colonne : (4, 1)
ligne : (1, 3)

somme : (4, 3)
[[11 21 31]
 [12 22 32]
 [13 23 33]
 [14 24 34]]


`(4, 1)` et `(1, 3)` deviennent `(4, 3)`. Les deux tableaux sont étirés en même
temps : la colonne sur 3 colonnes, la ligne sur 4 lignes.

C'est ce qu'on appelle le produit extérieur. Avec la multiplication, on obtient
directement une table de multiplication :

In [8]:
i = np.arange(1, 6).reshape(-1, 1) # colonne (5, 1)
j = np.arange(1, 6).reshape(1, -1) # ligne (1, 5)

print(i * j)

[[ 1  2  3  4  5]
 [ 2  4  6  8 10]
 [ 3  6  9 12 15]
 [ 4  8 12 16 20]
 [ 5 10 15 20 25]]


Une table 5×5, sans aucune boucle. C'est le remplacement vectorisé des boucles
imbriquées du notebook 03.

Le `reshape(-1, 1)` et `reshape(1, -1)` transforment un vecteur en colonne et en
ligne les deux formes complémentaires nécessaires au produit extérieur.

## 5. Le piège silencieux

C'est la section la plus importante du notebook. Le broadcasting ne se contente
pas d'échouer proprement : parfois il réussit là où on ne le voulait pas.

Prenons un exemple réaliste, tiré de la génération de données.

In [9]:
from sklearn.datasets import make_regression

np.random.seed(0)
x, y = make_regression(n_samples=1000, n_features=1, noise=10)

print('x :', x.shape)
print('y :', y.shape)

x : (1000, 1)
y : (1000,)


`x` a la forme `(1000, 1)` une matrice colonne. `y` a la forme `(1000,)` un
vecteur simple. C'est le `(n, 1)` contre `(n,)` du notebook 10.

Ces deux formes semblent équivalentes : mille valeurs dans les deux cas. On
pourrait vouloir calculer leur différence.

In [10]:
difference = x - y
print('x - y :', difference.shape)

x - y : (1000, 1000)


**`(1000, 1000)`.** Un million de valeurs là où on en attendait mille. Et aucune
erreur.

Voici ce qui s'est passé, en appliquant la règle de droite à gauche :

```
x :  (1000, 1)
y :  (      1000,)
```

NumPy aligne à droite et complète à gauche par des 1 :

```
x :  (1000, 1)
y :  (   1, 1000)
```

La dernière dimension : 1 et 1000, le 1 s'étire à 1000. La première : 1000 et 1,
le 1 s'étire à 1000. Résultat : `(1000, 1000)`.

Chaque valeur de `x` a été soustraite à chaque valeur de `y`. C'est
mathématiquement valide, donc pas d'erreur mais ce n'est jamais ce qu'on veut.

### Pourquoi c'est dangereux

Sur mille éléments, la matrice fait un million de cases : le calcul passe, juste
un peu lent. Le code continue, un `.mean()` plus loin donne un nombre plausible,
et le résultat est faux sans que rien ne l'indique.

Sur cent mille éléments, la matrice ferait dix milliards de cases là, la
mémoire sature et le noyau plante. Le bug se révèle, mais tard.

### La correction

Il faut donner à `y` la même forme que `x` :

In [11]:
y_correct = y.reshape(-1, 1)

print('y reshapé :', y_correct.shape)
print('x - y :', (x - y_correct).shape)

y reshapé : (1000, 1)
x - y : (1000, 1)


`(1000, 1)` moins `(1000, 1)` donne `(1000, 1)` l'addition case par case
attendue.

C'est exactement pour cette raison que le code de machine learning est truffé de
`reshape(-1, 1)` : pour forcer les formes à coïncider et éviter ce broadcasting
involontaire.

### Le réflexe

Avant toute opération entre deux tableaux, afficher les deux formes.

```python
print(x.shape, y.shape)
```

Si l'une est `(n, 1)` et l'autre `(n,)`, il y a un piège. Aligner les deux soit
en `(n, 1)`, soit en `(n,)` avec `.ravel()` avant de calculer.

C'est le réflexe qui aurait évité les erreurs de dimension rencontrées dans le
notebook SciPy avec `x0` et `minimize`.

## 6. Applications concrètes

Le broadcasting n'est pas qu'un piège : c'est l'outil qui rend NumPy si concis.

### Normaliser chaque colonne

Vu au notebook 12, mais maintenant on comprend pourquoi ça marche.

In [12]:
np.random.seed(0)
X = np.random.randint(0, 100, (5, 3)).astype(float)

moyennes = X.mean(axis=0) # (3,) : une par colonne
ecarts = X.std(axis=0) # (3,)

Z = (X - moyennes) / ecarts # (5,3) - (3,) : broadcasting

print('X shape :', X.shape)
print('moyennes shape:', moyennes.shape)
print('Z shape :', Z.shape)
print()
print('moyennes de Z :', Z.mean(axis=0).round(10))

X shape : (5, 3)
moyennes shape: (3,)
Z shape : (5, 3)

moyennes de Z : [0. 0. 0.]


`(5, 3)` moins `(3,)` : la règle complète `(3,)` en `(1, 3)`, puis étire sur les
5 lignes. Chaque ligne se voit soustraire le même vecteur de moyennes. Exactement
l'effet voulu.

### Normaliser chaque ligne : le rôle de keepdims

Pour diviser chaque ligne par sa somme, il faut cette fois `keepdims`.

In [13]:
X = np.random.randint(1, 10, (3, 4)).astype(float)

sans = X.sum(axis=1) # (3,)
avec = X.sum(axis=1, keepdims=True) # (3, 1)

print('sans keepdims :', sans.shape)
print('avec keepdims :', avec.shape)

sans keepdims : (3,)
avec keepdims : (3, 1)


In [14]:
# Avec keepdims : chaque ligne divisée par sa somme
resultat = X / X.sum(axis=1, keepdims=True)

print('sommes des lignes :', resultat.sum(axis=1))

sommes des lignes : [1. 1. 1.]


`(3, 4)` divisé par `(3, 1)` : la colonne des sommes s'étire sur les 4 colonnes.
Chaque ligne est divisée par sa propre somme, et somme donc à 1.

Sans `keepdims`, `(3, 4)` divisé par `(3,)` déclencherait le piège de la section
5 :

In [15]:
try:
    X / X.sum(axis=1) # (3,4) / (3,)
except ValueError as e:
    print('ValueError :', str(e)[:70])

ValueError : operands could not be broadcast together with shapes (3,4) (3,) 


Ici NumPy refuse, car `(3, 4)` et `(3,)` alignés à droite donnent 4 contre 3.
C'est une chance : l'erreur est visible. Mais sur un tableau carré `(3, 3)`, le
calcul passerait et donnerait un résultat faux d'où l'importance de `keepdims`.

### Une table de distances

Le broadcasting calcule toutes les distances entre points sans boucle.

In [16]:
points = np.array([0, 1, 4, 9])

distances = np.abs(points.reshape(-1, 1) - points.reshape(1, -1))

print('Table des distances :')
print(distances)

Table des distances :
[[0 1 4 9]
 [1 0 3 8]
 [4 3 0 5]
 [9 8 5 0]]


`(4, 1)` moins `(1, 4)` donne `(4, 4)` : chaque point moins chaque autre. La
diagonale est nulle la distance d'un point à lui-même.

C'est le fondement des k plus proches voisins, où l'on cherche les points les plus
proches d'un individu. Le broadcasting remplace une double boucle qui serait des
dizaines de fois plus lente.

### Colorer une image par condition

Le broadcasting s'applique aussi aux images du notebook 11.

In [17]:
# Une image factice (hauteur, largeur, RGB)
image = np.random.randint(0, 255, (4, 4, 3), dtype=np.uint8)

# Un facteur par canal : assombrir le rouge, garder vert, éclaircir bleu
facteurs = np.array([0.5, 1.0, 1.0]) # (3,)

resultat = (image * facteurs).clip(0, 255).astype(np.uint8)

print('image :', image.shape)
print('facteurs :', facteurs.shape)
print('resultat :', resultat.shape)

image : (4, 4, 3)
facteurs : (3,)
resultat : (4, 4, 3)


`(4, 4, 3)` multiplié par `(3,)` : le facteur s'applique au dernier axe, les trois
canaux. Chaque pixel voit son rouge, vert et bleu multipliés par les facteurs
correspondants.

`clip(0, 255)` évite le dépassement du type `uint8` vu au notebook 10.

C'est ainsi que fonctionnent les filtres de couleur, sans jamais parcourir les
pixels un à un.

## 7. Mémo

### La règle, en une phrase

En comparant les formes de droite à gauche, deux dimensions sont compatibles
si elles sont égales ou si l'une vaut 1. Une dimension à 1 est étirée.

### Exemples

| Forme A | Forme B | Résultat |
|---|---|---|
| `(2, 3)` | `(2, 3)` | `(2, 3)` |
| `(2, 3)` | scalaire | `(2, 3)` |
| `(2, 3)` | `(2, 1)` | `(2, 3)`, colonne étirée |
| `(2, 3)` | `(1, 3)` | `(2, 3)`, ligne étirée |
| `(4, 1)` | `(1, 3)` | `(4, 3)`, les deux étirés |
| `(2, 3)` | `(2, 2)` | erreur |
| `(1000, 1)` | `(1000,)` | `(1000, 1000)`, le piège |

### Les outils

| Besoin | Outil |
|---|---|
| Vecteur → colonne | `.reshape(-1, 1)` |
| Vecteur → ligne | `.reshape(1, -1)` |
| Garder l'axe après agrégation | `keepdims=True` |
| Colonne → vecteur | `.ravel()` ou `.squeeze()` |

### Les pièges

| Situation | Ce qui se passe |
|---|---|
| `(n, 1)` op `(n,)` | matrice `(n, n)`, sans erreur |
| `X / X.sum(axis=1)` | échoue ou fausse selon la forme |
| Oublier `keepdims` | broadcasting involontaire |
| Ne pas vérifier les formes | bug silencieux |

Le réflexe unique : `print(a.shape, b.shape)` avant toute opération.

## 8. Exercices

**Exercice 1**

Sans aucune boucle, construis la table de multiplication de 1 à 10 avec le
broadcasting. Puis la table d'addition. Vérifie les formes.

**Exercice 2**

On a une matrice `X` de forme `(100, 5)` 100 individus, 5 variables et un
vecteur `poids` de forme `(5,)`. Écris le score pondéré de chaque individu, c'est
la somme de ses variables multipliées par les poids.

Fais-le de deux façons : avec le broadcasting puis une somme, et avec un produit
matriciel. Vérifie que les résultats coïncident, et compare les formes obtenues.

**Exercice 3**

Ce code contient le piège silencieux du notebook :

```python
import numpy as np
prix = np.array([[10], [20], [30], [40]])
remises = np.array([0.1, 0.2, 0.5])
prix_finaux = prix - prix * remises
```

Affiche la forme de `prix_finaux`. Est-ce ce qu'on attend ? Explique ce que
NumPy a calculé, dans quel cas ce serait volontaire, et comment obtenir un prix
final par produit si c'était l'intention inverse.

## Pour continuer

C'est la fin du bloc NumPy. Tu disposes maintenant de tout ce qui fait tourner le
machine learning : créer des tableaux, les indexer, calculer dessus, et
comprendre les formes.

Le notebook suivant ouvre Matplotlib, pour visualiser ces données. Puis viendra
Pandas, qui pose une couche d'étiquettes par-dessus les tableaux NumPy mais
dessous, c'est toujours du NumPy, et le broadcasting continue d'opérer.

In [1]:
# Exercice 1

import numpy as np

i = np.arange(1, 11).reshape(-1, 1) # colonne (10, 1)
j = np.arange(1, 11).reshape(1, -1) # ligne   (1, 10)

table_mult = i * j
table_add = i + j

print('i :', i.shape, '  j :', j.shape)
print('table_mult :', table_mult.shape)
print()
print(table_mult)

i : (10, 1)   j : (1, 10)
table_mult : (10, 10)

[[  1   2   3   4   5   6   7   8   9  10]
 [  2   4   6   8  10  12  14  16  18  20]
 [  3   6   9  12  15  18  21  24  27  30]
 [  4   8  12  16  20  24  28  32  36  40]
 [  5  10  15  20  25  30  35  40  45  50]
 [  6  12  18  24  30  36  42  48  54  60]
 [  7  14  21  28  35  42  49  56  63  70]
 [  8  16  24  32  40  48  56  64  72  80]
 [  9  18  27  36  45  54  63  72  81  90]
 [ 10  20  30  40  50  60  70  80  90 100]]


In [2]:
v = np.arange(1, 11)
print('v * v :', (v * v).shape, '-> case par case, PAS une table')
print(v * v)

v * v : (10,) -> case par case, PAS une table
[  1   4   9  16  25  36  49  64  81 100]


In [3]:
print(table_add)
print('diagonale (i+i) :', np.diag(table_add))

[[ 2  3  4  5  6  7  8  9 10 11]
 [ 3  4  5  6  7  8  9 10 11 12]
 [ 4  5  6  7  8  9 10 11 12 13]
 [ 5  6  7  8  9 10 11 12 13 14]
 [ 6  7  8  9 10 11 12 13 14 15]
 [ 7  8  9 10 11 12 13 14 15 16]
 [ 8  9 10 11 12 13 14 15 16 17]
 [ 9 10 11 12 13 14 15 16 17 18]
 [10 11 12 13 14 15 16 17 18 19]
 [11 12 13 14 15 16 17 18 19 20]]
diagonale (i+i) : [ 2  4  6  8 10 12 14 16 18 20]


In [4]:
for i in range(1, 4):
    for j in range(1, 4):
        print(i * j)

1
2
3
2
4
6
3
6
9


In [5]:
# Exercice 2

In [6]:
rng = np.random.default_rng(0)

X = rng.normal(size=(100, 5)) # 100 individus, 5 variables
poids = np.array([0.1, 0.2, 0.3, 0.2, 0.2])

# Méthode 1 : broadcasting puis somme
scores_bc = (X * poids).sum(axis=1)

# Méthode 2 : produit matriciel
scores_mat = X @ poids

print('broadcasting :', scores_bc.shape)
print('produit matriciel :', scores_mat.shape)
print('identiques :', np.allclose(scores_bc, scores_mat))

broadcasting : (100,)
produit matriciel : (100,)
identiques : True


In [7]:
print('X * poids :', (X * poids).shape, '-> chaque variable pondérée')

X * poids : (100, 5) -> chaque variable pondérée


In [8]:
import time

N = 100_000
X = rng.normal(size=(N, 50))
poids = rng.normal(size=50)

debut = time.time(); (X * poids).sum(axis=1)
t1 = time.time() - debut
debut = time.time(); X @ poids
t2 = time.time() - debut

print(f'broadcasting + somme : {t1:.4f} s')
print(f'produit matriciel    : {t2:.4f} s')

broadcasting + somme : 0.0259 s
produit matriciel    : 0.0049 s


In [9]:
# Exercice 3

In [10]:
import numpy as np

prix = np.array([[10], [20], [30], [40]]) # (4, 1)
remises = np.array([0.1, 0.2, 0.5]) # (3,)

prix_finaux = prix - prix * remises
print('forme :', prix_finaux.shape)
print(prix_finaux)

forme : (4, 3)
[[ 9.  8.  5.]
 [18. 16. 10.]
 [27. 24. 15.]
 [36. 32. 20.]]


In [11]:
prix = np.array([[10], [20], [30], [40]])  # (4, 1)
remises = np.array([0.1, 0.2, 0.5, 0.3]).reshape(-1, 1) # (4, 1)

prix_finaux = prix * (1 - remises)
print('forme :', prix_finaux.shape)
print(prix_finaux)

forme : (4, 1)
[[ 9.]
 [16.]
 [15.]
 [28.]]


In [12]:
print('prix :', prix.shape, ' remises :', remises.shape)

prix : (4, 1)  remises : (4, 1)
